## Data Loading and Basic Statistical Analysis
Load multiple scoring results files and perform basic statistical analysis to obtain stable estimates given LLM stochasticity.

In [1]:
# Load and analyze multiple results files for stable statistics
import glob
import pandas as pd
import numpy as np

# Define the pattern for results files
result_path = 'results/circuit'
results_pattern = f'{result_path}/*/combined_results*.csv'
results_files = glob.glob(results_pattern)

print(f"Found {len(results_files)} results files:")
for file in results_files:
    print(f"  - {file}")

# Load all results files
all_results = []
for i, file in enumerate(results_files):
    df = pd.read_csv(file)
    df['sample_id'] = i + 1  # Add sample identifier
    df['file_source'] = file
    all_results.append(df)

if all_results:
    combined_df = pd.concat(all_results, ignore_index=True)
    print(f"\nCombined dataset: {len(combined_df)} total evaluations across {len(results_files)} samples")
    print(f"Columns: {list(combined_df.columns)}")
    
    # Basic statistics
    score_mean = combined_df['score'].mean()
    score_std = combined_df['score'].std()
    score_se = score_std / np.sqrt(len(combined_df))
    
    print(f"\nOverall score statistics:")
    print(f"Mean: {score_mean:.3f}")
    print(f"Std: {score_std:.3f}")
    print(f"Standard Error: {score_se:.3f}")
    print(f"Median: {combined_df['score'].median():.3f}")
    print(f"Min: {combined_df['score'].min():.1f}")
    print(f"Max: {combined_df['score'].max():.1f}")
else:
    print("No results files found. Please check the file pattern.")
    combined_df = pd.DataFrame()

Found 9 results files:
  - results/circuit/sample_3/combined_results_2.csv
  - results/circuit/sample_3/combined_results_1.csv
  - results/circuit/sample_3/combined_results_3.csv
  - results/circuit/sample_1/combined_results_2.csv
  - results/circuit/sample_1/combined_results_1.csv
  - results/circuit/sample_1/combined_results_3.csv
  - results/circuit/sample_2/combined_results_2.csv
  - results/circuit/sample_2/combined_results_1.csv
  - results/circuit/sample_2/combined_results_3.csv

Combined dataset: 3024 total evaluations across 9 samples
Columns: ['scorer', 'summariser', 'task_name', 'condition', 'score', 'judgment', 'answer', 'sample_id', 'file_source']

Overall score statistics:
Mean: 6.417
Std: 2.033
Standard Error: 0.037
Median: 7.000
Min: 1.0
Max: 10.0


# Analyse stability across samples for each condition-model-task combination

In [2]:
if not combined_df.empty:
    # Group by experimental conditions
    grouping_cols = ['summariser', 'condition', 'task_name', 'scorer']
    
    # Calculate statistics for each group across samples
    stability_stats = combined_df.groupby(grouping_cols)['score'].agg([
        'count', 'mean', 'std', 'median', 'min', 'max'
    ]).round(3)
    
    # Add standard error
    stability_stats['std_error'] = combined_df.groupby(grouping_cols)['score'].apply(
        lambda x: x.std() / np.sqrt(len(x)) if len(x) > 0 else np.nan
    ).round(3)
    
    # Add confidence intervals (95%)
    stability_stats['ci_lower'] = combined_df.groupby(grouping_cols)['score'].apply(
        lambda x: x.mean() - 1.96 * x.std() / np.sqrt(len(x)) if len(x) > 1 else x.mean()
    ).round(3)
    stability_stats['ci_upper'] = combined_df.groupby(grouping_cols)['score'].apply(
        lambda x: x.mean() + 1.96 * x.std() / np.sqrt(len(x)) if len(x) > 1 else x.mean()
    ).round(3)
    
    print("=== Stability Analysis Across Samples ===")
    print(f"Total unique condition combinations: {len(stability_stats)}")
    print(f"Combinations with multiple samples: {(stability_stats['count'] > 1).sum()}")
    
    # Show most variable combinations
    print("\n=== Most Variable Combinations (highest std) ===")
    most_variable = stability_stats[stability_stats['count'] > 1].nlargest(10, 'std')
    print(most_variable[['count', 'mean', 'std', 'std_error', 'ci_lower', 'ci_upper']])
    
    # Show most stable combinations
    print("\n=== Most Stable Combinations (lowest std, min 2 samples) ===")
    most_stable = stability_stats[stability_stats['count'] > 1].nsmallest(10, 'std')
    print(most_stable[['count', 'mean', 'std', 'std_error', 'ci_lower', 'ci_upper']])

=== Stability Analysis Across Samples ===
Total unique condition combinations: 336
Combinations with multiple samples: 336

=== Most Variable Combinations (highest std) ===
                                                                                      count  \
summariser                 condition            task_name scorer                              
deepseek-reasoner          lens_ap_global_local task_2_2  deepseek-reasoner               9   
                           lens_ap              task_3    o3-mini-2025-01-31              9   
                           lens_ap_global_local task_2_1  deepseek-reasoner               9   
                           rm_ap                task_3    claude-3-7-sonnet-20250219      9   
claude-3-7-sonnet-20250219 lens_ap_local        task_3    deepseek-reasoner               9   
deepseek-reasoner          rm_ap                task_2_1  deepseek-reasoner               9   
                           lens_two_cm          task_3    o3-mini-2

# Condition-level analysis with confidence intervals

In [3]:
if not combined_df.empty:
    print("=== Analysis by Condition (Aggregated across all other factors) ===")
    
    # Aggregate across models, tasks, and scorers for each condition
    condition_stats = combined_df.groupby(['condition', 'sample_id'])['score'].mean().reset_index()
    condition_summary = condition_stats.groupby('condition')['score'].agg([
        'count', 'mean', 'std', 'median'
    ]).round(3)
    
    # Add standard error
    condition_summary['std_error'] = condition_stats.groupby('condition')['score'].apply(
        lambda x: x.std() / np.sqrt(len(x)) if len(x) > 0 else np.nan
    ).round(3)
    
    # Add confidence intervals
    condition_summary['ci_lower'] = condition_stats.groupby('condition')['score'].apply(
        lambda x: x.mean() - 1.96 * x.std() / np.sqrt(len(x)) if len(x) > 1 else x.mean()
    ).round(3)
    condition_summary['ci_upper'] = condition_stats.groupby('condition')['score'].apply(
        lambda x: x.mean() + 1.96 * x.std() / np.sqrt(len(x)) if len(x) > 1 else x.mean()
    ).round(3)
    
    print(condition_summary.sort_values('mean', ascending=False))
    
    print("\n=== Analysis by Response Model ===")
    model_stats = combined_df.groupby(['summariser', 'sample_id'])['score'].mean().reset_index()
    model_summary = model_stats.groupby('summariser')['score'].agg([
        'count', 'mean', 'std', 'median'
    ]).round(3)
    
    # Add standard error
    model_summary['std_error'] = model_stats.groupby('summariser')['score'].apply(
        lambda x: x.std() / np.sqrt(len(x)) if len(x) > 0 else np.nan
    ).round(3)
    
    model_summary['ci_lower'] = model_stats.groupby('summariser')['score'].apply(
        lambda x: x.mean() - 1.96 * x.std() / np.sqrt(len(x)) if len(x) > 1 else x.mean()
    ).round(3)
    model_summary['ci_upper'] = model_stats.groupby('summariser')['score'].apply(
        lambda x: x.mean() + 1.96 * x.std() / np.sqrt(len(x)) if len(x) > 1 else x.mean()
    ).round(3)
    
    print(model_summary.sort_values('mean', ascending=False))
    
    print("\n=== Analysis by Task ===")
    task_stats = combined_df.groupby(['task_name', 'sample_id'])['score'].mean().reset_index()
    task_summary = task_stats.groupby('task_name')['score'].agg([
        'count', 'mean', 'std', 'median'
    ]).round(3)
    
    # Add standard error
    task_summary['std_error'] = task_stats.groupby('task_name')['score'].apply(
        lambda x: x.std() / np.sqrt(len(x)) if len(x) > 0 else np.nan
    ).round(3)
    
    task_summary['ci_lower'] = task_stats.groupby('task_name')['score'].apply(
        lambda x: x.mean() - 1.96 * x.std() / np.sqrt(len(x)) if len(x) > 1 else x.mean()
    ).round(3)
    task_summary['ci_upper'] = task_stats.groupby('task_name')['score'].apply(
        lambda x: x.mean() + 1.96 * x.std() / np.sqrt(len(x)) if len(x) > 1 else x.mean()
    ).round(3)
    
    print(task_summary.sort_values('mean', ascending=False))

=== Analysis by Condition (Aggregated across all other factors) ===
                      count   mean    std  median  std_error  ci_lower  \
condition                                                                
lens_np_global_local      9  7.213  0.507   7.083      0.169     6.882   
lens_ap_local             9  6.963  0.202   7.000      0.067     6.831   
lens_np_local             9  6.801  0.104   6.792      0.035     6.733   
lens_np_global            9  6.731  0.313   6.792      0.104     6.527   
rm_np                     9  6.662  0.343   6.667      0.114     6.438   
lens_np                   9  6.620  0.539   6.875      0.180     6.268   
lens_ap_global            9  6.509  0.194   6.417      0.065     6.382   
lens_ap                   9  6.366  0.217   6.375      0.072     6.224   
lens_ap_global_local      9  6.356  0.548   6.375      0.183     5.999   
rm_global                 9  6.356  0.389   6.375      0.130     6.103   
lens_two_cm               9  6.231  0.310   

# Calculate score percentiles by task for human trial selection

In [4]:
if not combined_df.empty:
    print("=== Score Percentiles by Task ===")
    
    # Calculate percentiles for each task
    percentiles = [10, 25, 50, 75, 90, 95]
    
    for task in sorted(combined_df['task_name'].unique()):
        task_scores = combined_df[combined_df['task_name'] == task]['score']
        print(f"\nTask: {task}")
        print(f"  Count: {len(task_scores)}")
        print(f"  Mean: {task_scores.mean():.3f}")
        print(f"  Percentiles:")
        
        for p in percentiles:
            value = np.percentile(task_scores, p)
            print(f"    {p}th: {value:.3f}")
    
    # Create a comprehensive percentile table
    print("\n=== Percentile Table for All Tasks ===")
    task_percentiles = {}
    
    for task in sorted(combined_df['task_name'].unique()):
        task_scores = combined_df[combined_df['task_name'] == task]['score']
        task_percentiles[task] = {
            'count': len(task_scores),
            'mean': task_scores.mean(),
            'std': task_scores.std()
        }
        
        for p in percentiles:
            task_percentiles[task][f'p{p}'] = np.percentile(task_scores, p)
    
    # Convert to DataFrame for better display
    percentile_df = pd.DataFrame.from_dict(task_percentiles, orient='index').round(3)
    print(percentile_df)
    
    # Save percentile analysis
    percentile_df.to_csv(f'{result_path}/lens_task_percentiles.csv')
    print(f"\nPercentile analysis saved to: lens_task_percentiles.csv")
    
    # Identify candidates for human evaluation at different quality levels
    print("\n=== Suggested Response Selection for Human Trial ===")
    print("High quality (≥75th percentile):")
    high_quality = combined_df[combined_df.groupby('task_name')['score'].transform(lambda x: x >= x.quantile(0.75))]
    print(f"  {len(high_quality)} responses across {high_quality['task_name'].nunique()} tasks")
    
    print("Medium quality (25th-75th percentile):")
    medium_quality = combined_df[
        (combined_df.groupby('task_name')['score'].transform(lambda x: x >= x.quantile(0.25))) &
        (combined_df.groupby('task_name')['score'].transform(lambda x: x < x.quantile(0.75)))
    ]
    print(f"  {len(medium_quality)} responses across {medium_quality['task_name'].nunique()} tasks")
    
    print("Low quality (<25th percentile):")
    low_quality = combined_df[combined_df.groupby('task_name')['score'].transform(lambda x: x < x.quantile(0.25))]
    print(f"  {len(low_quality)} responses across {low_quality['task_name'].nunique()} tasks")

=== Score Percentiles by Task ===

Task: task_1
  Count: 756
  Mean: 7.226
  Percentiles:
    10th: 6.000
    25th: 6.750
    50th: 7.000
    75th: 8.000
    90th: 9.000
    95th: 9.000

Task: task_2_1
  Count: 756
  Mean: 5.152
  Percentiles:
    10th: 3.000
    25th: 4.000
    50th: 5.000
    75th: 6.000
    90th: 7.000
    95th: 8.000

Task: task_2_2
  Count: 756
  Mean: 5.276
  Percentiles:
    10th: 3.000
    25th: 4.000
    50th: 5.000
    75th: 7.000
    90th: 8.000
    95th: 8.000

Task: task_3
  Count: 756
  Mean: 8.013
  Percentiles:
    10th: 6.000
    25th: 7.000
    50th: 9.000
    75th: 9.000
    90th: 9.000
    95th: 10.000

=== Percentile Table for All Tasks ===
          count   mean    std  p10   p25  p50  p75  p90   p95
task_1      756  7.226  1.396  6.0  6.75  7.0  8.0  9.0   9.0
task_2_1    756  5.152  1.702  3.0  4.00  5.0  6.0  7.0   8.0
task_2_2    756  5.276  1.780  3.0  4.00  5.0  7.0  8.0   8.0
task_3      756  8.013  1.555  6.0  7.00  9.0  9.0  9.0  10.0

Pe

# Save the combined results and summary statistics

In [5]:
if not combined_df.empty:
    # Save the combined results
    output_file = f'{result_path}/lens_multi_sample_analysis.csv'
    combined_df.to_csv(output_file, index=False)
    
    # Save stability statistics
    if 'stability_stats' in locals():
        stability_stats.to_csv(f'{result_path}/lens_stability_analysis.csv')
        
    print(f"\nResults saved:")
    print(f"- Combined data: {output_file}")
    print(f"- Stability analysis: lens_stability_analysis.csv")
    
    # Summary statistics table
    print("\n=== Multi-Sample Summary ===")
    print(f"Number of samples: {len(results_files)}")
    print(f"Total evaluations: {len(combined_df)}")
    print(f"Unique conditions: {combined_df['condition'].nunique()}")
    print(f"Unique models: {combined_df['summariser'].nunique()}")
    print(f"Unique tasks: {combined_df['task_name'].nunique()}")
    print(f"Unique scorers: {combined_df['scorer'].nunique()}")
    
    # Calculate overall variability
    overall_sample_means = combined_df.groupby('sample_id')['score'].mean()
    overall_mean = overall_sample_means.mean()
    overall_std = overall_sample_means.std()
    overall_se = overall_std / np.sqrt(len(overall_sample_means))
    
    print(f"\nOverall sample-to-sample variability:")
    print(f"  Mean across samples: {overall_mean:.3f}")
    print(f"  Std across samples: {overall_std:.3f}")
    print(f"  Standard Error: {overall_se:.3f}")
    print(f"  95% CI: [{overall_mean - 1.96*overall_std:.3f}, "
          f"{overall_mean + 1.96*overall_std:.3f}]")


Results saved:
- Combined data: results/circuit/lens_multi_sample_analysis.csv
- Stability analysis: lens_stability_analysis.csv

=== Multi-Sample Summary ===
Number of samples: 9
Total evaluations: 3024
Unique conditions: 14
Unique models: 2
Unique tasks: 4
Unique scorers: 3

Overall sample-to-sample variability:
  Mean across samples: 6.417
  Std across samples: 0.083
  Standard Error: 0.028
  95% CI: [6.255, 6.579]
